<a href="https://colab.research.google.com/github/Tiyinolalekan/Deep_Learning_Image_Classifier/blob/main/Training_Notes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***THROW AWAY NOTES:***

_______________________
Template for class weigh imbalance handling

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

data_dir = "/content/clothes/clothing_dataset"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Data
tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(f"{data_dir}/train", tf)
val_ds = datasets.ImageFolder(f"{data_dir}/val", tf)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=32)

# 2. Class weights (same formula as before)
counts = torch.bincount(torch.tensor(train_ds.targets))
weights = len(train_ds) / (len(counts) * counts.float())

# 3. Pretrained model with a new final layer
model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, len(train_ds.classes))
model = model.to(device)

# 4. Loss with weights, and optimizer
loss_fn = nn.CrossEntropyLoss(weight=weights.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# 5. Training loop
for epoch in range(10):
    model.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
    print(f"epoch {epoch+1}: val acc {correct / len(val_ds):.3f}")